In [4]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
import os
from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray, Camera


In [5]:

# Load scene
scene = load_scene("chalmers_entrance_test/Chalmers_entrance.xml")

# Define multiple waypoints for a realistic path around obstacles
waypoints = [
    [-50, 30, 1.2],     # start
    [-14, 35, 1.2],    # move
    [10, 1.4, 1.2],   # move
    [44, -10, 1.2],    # move
    [55, -30, 1.2],   # end
]


In [6]:
# Function to create a smooth path between waypoints
def generate_smooth_path(waypoints, steps_per_segment=20):
    path = []
    for i in range(len(waypoints) - 1):
        segment = np.linspace(waypoints[i], waypoints[i+1], steps_per_segment).tolist()
        path.extend(segment)
    return path



In [7]:
# Generate smooth path through all waypoints
receiver_path = generate_smooth_path(waypoints, steps_per_segment=20)

# Initialize transmitter and receiver
scene.tx_array = PlanarArray(num_rows=4, num_cols=4, vertical_spacing=0.5, horizontal_spacing=0.5, pattern="tr38901", polarization="V")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, vertical_spacing=0.5, horizontal_spacing=0.5, pattern="iso", polarization="V")

tx = Transmitter("tx", [1.5, 36, 16], [0.0, 0.0, 0.0])
scene.add(tx)

rx = Receiver("rx", receiver_path[0], [0.0, 0.0, 0.0])
scene.add(rx)



In [13]:
# Add camera
my_cam = Camera("my_cam2", position=[150, 61, 90], look_at=[19, 3, 1.5])
scene.add(my_cam)

# Create directory for frames
os.makedirs("frames", exist_ok=True)

# Iterate through time steps
frame_files = []
for t in range(len(receiver_path)):
    # Update receiver position
    rx.position = receiver_path[t]
    
    # Compute coverage map and paths
    cm = scene.coverage_map()
    paths = scene.compute_paths()
    
    # Render frame
    frame_filename = f"frames/frame_{t:03d}.png"

    # Render the scene from the camera's viewpoint
    fig = scene.render("my_cam2", paths=paths, coverage_map=cm)

    # Save the rendered figure as an image
    fig.savefig(frame_filename)
    plt.close(fig)  # Close the figure to free memory

    frame_files.append(frame_filename)


In [14]:
# Create video
video_filename = "multi_waypoint_receiver.mp4"
frame = cv2.imread(frame_files[0])
h, w, _ = frame.shape
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video = cv2.VideoWriter(video_filename, fourcc, 5, (w, h))  # 5 FPS

for frame_file in frame_files:
    frame = cv2.imread(frame_file)
    video.write(frame)

video.release()
print(f"Video saved as {video_filename}")

Video saved as multi_waypoint_receiver.mp4
